# Membership Churn Analysis

## Notebook 03: Missingness & Data Quality Validation

---

**Author:** D. 
**Date:** March 3, 2026 
**Dataset:** churn_t_db.csv (2.1GB, 18.4M rows)

---

### Dependencies

In [0]:
# ═══════════════════════════════════════════════════════════════
# Dependencies
# ═══════════════════════════════════════════════════════════════

import sys
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType
import pandas as pd

print("Dependencies loaded successfully")

Dependencies loaded successfully


## 1.0 Setup & Configuration

### 1.1 Environment Verification

**CONTEXT**

To establish the foundational environment for the missingness and data quality validation analysis. This includes verifying Spark availability, confirming access to the Silver table produced in Notebook 02, and validating the cleaned dataset before any missingness analysis begins.

**PURPOSE**

To ensure:
1. PySpark environment is properly initialised
2. The Silver table is accessible at the expected Unity Catalog volume path
3. The cleaned dataset retains the expected 18,461,480 rows and 14 columns
4. Confident progression to missingness analysis

**STEP**

Confirm that the Silver table exists at the expected path, verify Spark environment initialisation, and validate row and column counts against the baseline established in Notebook 02.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 1.1 Environment Verification
# ═══════════════════════════════════════════════════════════════

# Check Spark version
print(f"Spark Version: {spark.version}")
print("-" * 70)

# Verify Silver table exists
file_info = dbutils.fs.ls("/Volumes/workspace/rcn_churn/silver/")
display(file_info)

# Confirm Silver table path
silver_path = "/Volumes/workspace/rcn_churn/silver/churn_cleaned/"
print(f"\nSilver table path: {silver_path}")

Spark Version: 4.1.0
----------------------------------------------------------------------


path,name,size,modificationTime
dbfs:/Volumes/workspace/rcn_churn/silver/churn_cleaned/,churn_cleaned/,0,1773331809856



Silver table path: /Volumes/workspace/rcn_churn/silver/churn_cleaned/


**RESULT**

The Silver table is accessible at `/Volumes/workspace/rcn_churn/silver/churn_cleaned/`. PySpark environment is initialised and operational. Environment is validated and ready for data loading.

**Status:** ✓ Pass

### 1.2 Data Loading

**CONTEXT**

Notebook 03 loads directly from the Silver table produced in Notebook 02, which contains the fully cleaned and transformed dataset. This ensures all missingness analysis operates on validated, consistently structured data with cleaning flags intact for reference.


**PURPOSE**

To ensure:
1. The Silver table loads successfully with the expected row and column count
2. The 14-column schema from Notebook 02 is preserved and consistent
3. Cleaning flags and geo flags are present and accessible
4. A verified baseline exists before any missingness analysis is applied


**STEP**

Load the Silver table into a PySpark DataFrame and confirm row count, column count, and schema against the baseline established in Notebook 02.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 1.2 Data Loading
# ═══════════════════════════════════════════════════════════════

# Define Silver table path
silver_path = "/Volumes/workspace/rcn_churn/silver/churn_cleaned/"

# Load Silver table
df = spark.read.format("delta").load(silver_path)

# Confirm row and column count
row_count = df.count()
col_count = len(df.columns)

print(f"Rows    : {row_count:,}")
print(f"Columns : {col_count}")
print("-" * 70)

# Display schema
df.printSchema()

Rows    : 18,461,480
Columns : 14
----------------------------------------------------------------------
root
 |-- _c0: integer (nullable = true)
 |-- CM_snapshot_date: date (nullable = true)
 |-- Int_nurse: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- MemCategory: string (nullable = true)
 |-- CatName: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- YoB: integer (nullable = true)
 |-- MemSectorType: string (nullable = true)
 |-- YoJ: integer (nullable = true)
 |-- q_members_t: double (nullable = true)
 |-- q_leavers_t: double (nullable = true)
 |-- cleaning_flag: string (nullable = true)
 |-- geo_flag: string (nullable = true)



**RESULT**

The Silver table loaded successfully with 18,461,480 rows and 14 columns, consistent with the baseline established in Notebook 02. The schema is preserved as expected, with `cleaning_flag` and `geo_flag` columns confirmed present for reference throughout the missingness analysis. `YoB` and `YoJ` are correctly stored as `integer` type confirming the type corrections from Notebook 02 are persisted in the Silver table.

**Status:** ✓ Pass

### 1.3 Configuration & Constants


**CONTEXT**

To centralise all notebook configuration in a single location, establishing path constants and analytical thresholds that will be referenced throughout the missingness and data quality validation pipeline.

**PURPOSE**

To ensure:
1. All file paths are defined consistently and referenced from a single source
2. Missingness thresholds are explicitly documented and transparent
3. Any future changes to configuration require a single update point
4. Confident progression to missingness analysis

**STEP**

Define all path constants and configuration variables required for Notebook 03's missingness and data quality validation pipeline.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 1.3 Configuration & Constants
# ═══════════════════════════════════════════════════════════════

# ── Paths ──────────────────────────────────────────────────────
SILVER_PATH = "/Volumes/workspace/rcn_churn/silver/churn_cleaned/"
GOLD_PATH   = "/Volumes/workspace/rcn_churn/gold/"

# ── Missingness Thresholds ─────────────────────────────────────
LOW_MISSING    = 0.01    # < 1% missing
MEDIUM_MISSING = 0.05    # 1% - 5% missing
HIGH_MISSING   = 0.10    # > 5% missing

# ── Run Metadata ───────────────────────────────────────────────
RUN_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Configuration loaded successfully")
print("-" * 70)
print(f"Silver Path : {SILVER_PATH}")
print(f"Gold Path   : {GOLD_PATH}")
print(f"Run Time    : {RUN_TS}")

Configuration loaded successfully
----------------------------------------------------------------------
Silver Path : /Volumes/workspace/rcn_churn/silver/churn_cleaned/
Gold Path   : /Volumes/workspace/rcn_churn/gold/
Run Time    : 2026-03-12 16:10:30


**RESULT**

All configuration constants loaded successfully. File paths and missingness thresholds are centralised and ready to be referenced throughout the missingness and data quality validation pipeline.

**Status:** ✓ Pass

## 2.0 Missingness Assessment

### 2.1 Overall Missingness Profile

**CONTEXT**

A comprehensive missingness profile establishes the baseline for all subsequent missingness analysis. Notebook 01 identified missingness across several columns, with `q_leavers_t` carrying a 98.9% null rate by design and `MemSectorType` carrying a 7.78% null rate requiring investigation. Notebook 02 introduced additional nulls through the cleaning pipeline, specifically in `YoB` where outliers and age violations were remediated by nulling invalid values. A fresh missingness profile against the Silver table captures the complete post-cleaning null landscape before formal classification begins.

**PURPOSE**

To ensure:
1. A complete and accurate null count is established for every column in the Silver table
2. Post-cleaning missingness is compared against Notebook 01 baseline findings
3. Columns are categorised by missingness severity to prioritise analytical treatment
4. A transparent foundation exists for the MCAR/MAR/MNAR classification in subsequent sections

**STEP**

Calculate the null count and null percentage for every column in the Silver table. Categorise each column by missingness severity using the defined thresholds of low, medium, and high. Compare findings against the Notebook 01 baseline to confirm the impact of Notebook 02 cleaning transformations on the null landscape.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.1 Overall Missingness Profile
# ═══════════════════════════════════════════════════════════════

total_rows = df.count()

# Exclude flag columns from missingness assessment
flag_columns = ["cleaning_flag", "geo_flag"]
analysis_columns = [c for c in df.columns if c not in flag_columns]

# Calculate null count and percentage for analytical columns
print("Overall Missingness Profile (Silver Table):")
print("-" * 70)

missingness_data = []
for col_name in analysis_columns:
    null_count = df.filter(F.col(col_name).isNull()).count()
    null_pct = (null_count / total_rows) * 100
    if null_pct == 0:
        severity = "None"
    elif null_pct < 1:
        severity = "Low"
    elif null_pct < 5:
        severity = "Medium"
    else:
        severity = "High"
    missingness_data.append((col_name, null_count, round(null_pct, 4), severity))

missingness_df = spark.createDataFrame(
    missingness_data,
    ["column", "null_count", "null_pct", "severity"]
)

missingness_df.orderBy(F.col("null_pct").desc()).show(20, truncate=False)

# Flag column summary
print("Flag Column Summary (null = clean record, expected by design):")
print("-" * 70)
for flag_col in flag_columns:
    flagged = df.filter(F.col(flag_col).isNotNull()).count()
    print(f"   {flag_col}: {flagged:,} flagged records "
          f"({round(flagged/total_rows*100, 4)}% of dataset)")

Overall Missingness Profile (Silver Table):
----------------------------------------------------------------------
+----------------+----------+--------+--------+
|column          |null_count|null_pct|severity|
+----------------+----------+--------+--------+
|q_leavers_t     |18259005  |98.9033 |High    |
|MemSectorType   |1436695   |7.7821  |High    |
|YoB             |275407    |1.4918  |Medium  |
|YoJ             |39        |2.0E-4  |Low     |
|_c0             |0         |0.0     |None    |
|CM_snapshot_date|0         |0.0     |None    |
|Int_nurse       |0         |0.0     |None    |
|Region          |0         |0.0     |None    |
|MemCategory     |0         |0.0     |None    |
|CatName         |0         |0.0     |None    |
|Branch          |0         |0.0     |None    |
|q_members_t     |0         |0.0     |None    |
+----------------+----------+--------+--------+

Flag Column Summary (null = clean record, expected by design):
-----------------------------------------------------

**RESULT**

The post-cleaning missingness profile confirms four columns with non-zero null rates across the 12 analytical columns. `q_leavers_t` retains its 98.90% null rate by design, representing active members with no leaver event recorded in that snapshot period. `MemSectorType` carries a 7.78% null rate, consistent with the Notebook 02 investigation which identified this as a combination of structurally expected nulls for non-active employment categories and missing at random nulls for ambiguous categories. `YoB` now carries a 1.49% null rate, increased from the Notebook 01 baseline as a direct result of the Notebook 02 cleaning pipeline nulling confirmed errors, default placeholders, and age violations. `YoJ` retains a negligible 0.0002% null rate across 39 records.

The flag column summary confirms 271,199 records carry a `cleaning_flag` representing 1.47% of the dataset, and 2 records carry a `geo_flag`. Null values in both flag columns are expected and indicate clean records requiring no remediation.

Seven columns retain zero missingness, confirming the structural integrity of the core dimensional and membership count fields.

**Status:** ✓ Pass

### 2.2 Missingness Classification (MCAR/MAR/MNAR)

**CONTEXT**

Following the overall missingness profile in Section 2.1, each column with non-zero missingness requires formal classification to determine the nature and analytical implications of its null values. The three classification types, Missing Completely At Random (MCAR), Missing At Random (MAR), and Missing Not At Random (MNAR), carry different implications for how missing values should be treated in downstream analysis. Notebook 02 investigations provided strong evidence for the classification of several columns, particularly `MemSectorType` and `YoB`, which are formalised here.

**PURPOSE**

To ensure:
1. Each column with non-zero missingness is formally classified as MCAR, MAR, MNAR, or Not Applicable
2. Classifications are evidence-based, drawing on findings from Notebooks 01 and 02
3. Analytical implications of each classification are clearly documented
4. Downstream notebooks apply appropriate treatment for each missing value type

**STEP**

Systematically evaluate each column with non-zero missingness against the MCAR, MAR, and MNAR criteria. Produce a structured classification table summarising the column, null count, null percentage, classification, justification, and recommended analytical treatment for each.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.2 Missingness Classification (MCAR/MAR/MNAR)
# ═══════════════════════════════════════════════════════════════

# Build structured classification table
classification_data = [
    (
        "q_leavers_t",
        18259005,
        98.90,
        "MCAR",
        "Null indicates active member with no leaver event in snapshot period. "
        "Missingness is by design and carries no analytical bias.",
        "Retain as null. Use IS NULL as active member filter in churn calculations."
    ),
    (
        "MemSectorType",
        1436695,
        7.78,
        "MAR / Not Applicable",
        "53.1% of nulls belong to definitively non-active employment categories "
        "where sector is not applicable by definition. Remaining 46.9% relates "
        "to ambiguous categories including Nurse Full, Student, and Life Member "
        "where missingness is related to membership category (observed variable).",
        "Retain as null. Exclude from sector-based segmentation or treat as "
        "separate Unknown category in aggregations."
    ),
    (
        "YoB",
        275407,
        1.49,
        "MAR",
        "Nulls introduced via Notebook 02 cleaning pipeline. YOB_DEFAULT_PLACEHOLDER "
        "records (YoB = 1900) suggest missingness is related to membership vintage "
        "and system of registration, both observed variables. No evidence of "
        "self-selection bias in the missing values.",
        "Exclude null YoB rows from age-based cohort analysis. "
        "Document as limitation in dissertation methodology."
    ),
    (
        "YoJ",
        39,
        0.0002,
        "MCAR",
        "39 records with null YoJ represent 0.0002% of the dataset. "
        "No systematic pattern identified in Notebook 01. "
        "Analytically negligible.",
        "Exclude from tenure-based calculations. No further treatment required."
    )
]

classification_df = spark.createDataFrame(
    classification_data,
    ["column", "null_count", "null_pct", "classification",
     "justification", "analytical_treatment"]
)

print("Missingness Classification Table:")
print("-" * 70)
classification_df.select(
    "column", "null_count", "null_pct", "classification"
).show(truncate=False)

print("Detailed Classification:")
print("-" * 70)
for row in classification_df.collect():
    print(f"Column         : {row['column']}")
    print(f"Null Count     : {row['null_count']:,}")
    print(f"Null Pct       : {row['null_pct']}%")
    print(f"Classification : {row['classification']}")
    print(f"Justification  : {row['justification']}")
    print(f"Treatment      : {row['analytical_treatment']}")
    print("-" * 70)

Missingness Classification Table:
----------------------------------------------------------------------
+-------------+----------+--------+--------------------+
|column       |null_count|null_pct|classification      |
+-------------+----------+--------+--------------------+
|q_leavers_t  |18259005  |98.9    |MCAR                |
|MemSectorType|1436695   |7.78    |MAR / Not Applicable|
|YoB          |275407    |1.49    |MAR                 |
|YoJ          |39        |2.0E-4  |MCAR                |
+-------------+----------+--------+--------------------+

Detailed Classification:
----------------------------------------------------------------------
Column         : q_leavers_t
Null Count     : 18,259,005
Null Pct       : 98.9%
Classification : MCAR
Justification  : Null indicates active member with no leaver event in snapshot period. Missingness is by design and carries no analytical bias.
Treatment      : Retain as null. Use IS NULL as active member filter in churn calculations.
----

**RESULT**

Four columns with non-zero missingness have been formally classified. `q_leavers_t` and `YoJ` are classified as MCAR, both analytically negligible with null carrying a deliberate design meaning for `q_leavers_t` and a 0.0002% impact for `YoJ`. `YoB` is classified as MAR, with missingness related to membership system vintage and registration origin rather than the birth year value itself, introducing no systematic bias into age-based analysis provided null rows are explicitly excluded. `MemSectorType` carries a dual classification of MAR and Not Applicable, reflecting the two distinct null populations identified in Notebook 02, the structurally expected nulls for non-active employment categories and the ambiguous nulls related to membership category. No columns were classified as MNAR, indicating that the dataset carries no evidence of self-selection bias in its missing values, a finding that strengthens the overall analytical validity of the dataset.

**Status:** ✓ Pass

### 2.3 Temporal Missingness Patterns

**CONTEXT**

While Section 2.1 established the overall null counts and Section 2.2 classified their nature, understanding how missingness behaves across the 60-month observation period adds a critical temporal dimension to the analysis. A column whose nulls are evenly distributed across time behaves differently analytically to one whose nulls are concentrated in specific periods, the former suggesting a structural characteristic of the data collection system and the latter suggesting a process change or system migration at a specific point in time.

**PURPOSE**

To ensure:
1. Temporal distribution of nulls is understood for all columns with non-zero missingness
2. Any step changes or trend patterns in missingness over time are identified and documented
3. Findings either confirm or challenge the MCAR/MAR/MNAR classifications from Section 2.2
4. Downstream temporal analysis accounts for any periods with elevated missingness

**STEP**

Plot the monthly null count for `YoB`, `YoJ`, and `MemSectorType` across all 60 snapshots. Examine for step changes, trends, or seasonal patterns that may indicate system or process changes. `q_leavers_t` is excluded as its temporal null pattern was confirmed by design in Section 2.2.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.3 Temporal Missingness Patterns
# ═══════════════════════════════════════════════════════════════

# Columns to analyse (excluding q_leavers_t - null by design)
temporal_cols = ["YoB", "YoJ", "MemSectorType"]

for col_name in temporal_cols:
    print(f"Temporal Null Pattern: {col_name}")
    print("-" * 70)
    df.groupBy("CM_snapshot_date") \
      .agg(
          F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0))
           .alias("null_count"),
          F.count("*").alias("total_rows"),
          F.round(
              F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)) /
              F.count("*") * 100, 4
          ).alias("null_pct")
      ) \
      .orderBy("CM_snapshot_date") \
      .show(60, truncate=False)

Temporal Null Pattern: YoB
----------------------------------------------------------------------
+----------------+----------+----------+--------+
|CM_snapshot_date|null_count|total_rows|null_pct|
+----------------+----------+----------+--------+
|2021-01-01      |5237      |284197    |1.8427  |
|2021-02-01      |5201      |286668    |1.8143  |
|2021-03-01      |5182      |288309    |1.7974  |
|2021-04-01      |5167      |290440    |1.779   |
|2021-05-01      |5131      |290890    |1.7639  |
|2021-06-01      |5108      |291313    |1.7534  |
|2021-07-01      |5083      |291633    |1.7429  |
|2021-08-01      |5060      |291910    |1.7334  |
|2021-09-01      |5050      |291734    |1.731   |
|2021-10-01      |5031      |292540    |1.7198  |
|2021-11-01      |5006      |292038    |1.7142  |
|2021-12-01      |4985      |291272    |1.7115  |
|2022-01-01      |4960      |290027    |1.7102  |
|2022-02-01      |4930      |291396    |1.6919  |
|2022-03-01      |4898      |292591    |1.674   |
|2

#### 2.3.1 YoB April 2022 Anomaly Investigation

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.3.1 YoB April 2022 Anomaly Investigation
# ═══════════════════════════════════════════════════════════════

# High level comparison across three months
print("YoB Null Summary: March, April, May 2022:")
print("-" * 70)
df.filter(
    F.col("CM_snapshot_date").isin(
        "2022-03-01", "2022-04-01", "2022-05-01"
    )
).groupBy("CM_snapshot_date") \
 .agg(
     F.sum(F.when(F.col("YoB").isNull(), 1).otherwise(0)).alias("total_nulls"),
     F.count("*").alias("total_rows")
 ).orderBy("CM_snapshot_date") \
 .show(truncate=False)

# April 2022 null YoB breakdown by Region
print("April 2022 YoB Null by Region:")
print("-" * 70)
df.filter(
    (F.col("CM_snapshot_date") == "2022-04-01") &
    F.col("YoB").isNull()
).groupBy("Region") \
 .agg(F.count("*").alias("null_count"),
      F.sum("q_members_t").alias("total_members")) \
 .orderBy(F.col("null_count").desc()) \
 .show(truncate=False)

# April 2022 null YoB breakdown by MemCategory
print("April 2022 YoB Null by MemCategory:")
print("-" * 70)
df.filter(
    (F.col("CM_snapshot_date") == "2022-04-01") &
    F.col("YoB").isNull()
).groupBy("MemCategory") \
 .agg(F.count("*").alias("null_count"),
      F.sum("q_members_t").alias("total_members")) \
 .orderBy(F.col("null_count").desc()) \
 .show(truncate=False)

# April 2022 null YoB breakdown by cleaning_flag
print("April 2022 YoB Null by Cleaning Flag:")
print("-" * 70)
df.filter(
    (F.col("CM_snapshot_date") == "2022-04-01") &
    F.col("YoB").isNull()
).groupBy("cleaning_flag") \
 .agg(F.count("*").alias("null_count"),
      F.sum("q_members_t").alias("total_members")) \
 .orderBy(F.col("null_count").desc()) \
 .show(truncate=False)

YoB Null Summary: March, April, May 2022:
----------------------------------------------------------------------
+----------------+-----------+----------+
|CM_snapshot_date|total_nulls|total_rows|
+----------------+-----------+----------+
|2022-03-01      |4898       |292591    |
|2022-04-01      |7812       |294946    |
|2022-05-01      |4856       |294155    |
+----------------+-----------+----------+

April 2022 YoB Null by Region:
----------------------------------------------------------------------
+----------------------+----------+------------------+
|Region                |null_count|total_members     |
+----------------------+----------+------------------+
|South East            |1022      |3806.413301662638 |
|London                |844       |3355.1068883609973|
|North West            |822       |3723.2779097386706|
|Scotland              |753       |3221.4964370546   |
|West Midlands         |726       |3019.5961995249068|
|South West            |725       |2782.0665083135

**RESULT**

The April 2022 snapshot contains 7,812 null `YoB` records, representing a near doubling of the expected monthly null count of approximately 4,900. The spike is isolated to a single month, with March 2022 at 4,898 and May 2022 returning to 4,856, confirming a one-off event rather than a structural shift.

The cleaning flag breakdown reveals two distinct contributing populations. `YOB_DEFAULT_PLACEHOLDER` records account for 4,795 nulls (61.4%), representing legacy system default records concentrated in the `Nurse member` category and distributed across all 13 regions proportionally. The remaining 2,998 records carry no cleaning flag, indicating genuinely missing `YoB` values present in the Bronze table prior to any cleaning transformations. The geographic and categorical distribution shows no single region or membership category as the sole driver, suggesting a broad rather than targeted event.

The timing coincides with the UK government's full implementation of the Living with COVID plan in April 2022, when healthcare organisations were managing significant administrative backlogs and potential membership reactivations following pandemic-related disruptions. Returning or rejoining members may not have had birth year captured at re-registration, contributing to the genuine null increase. The elevated `YOB_DEFAULT_PLACEHOLDER` count suggests a batch of legacy member records were also processed or updated in that period.

**Status:** ⚠️ Investigate

#### 2.3.2 April 2022 Broader Anomaly Check

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.3.2 April 2022 Broader Anomaly Check
# ═══════════════════════════════════════════════════════════════

# Check membership count stability around April 2022
print("Membership Count Stability: Q1-Q2 2022:")
print("-" * 70)
df.filter(
    F.col("CM_snapshot_date").isin(
        "2022-01-01", "2022-02-01", "2022-03-01",
        "2022-04-01", "2022-05-01", "2022-06-01"
    )
).groupBy("CM_snapshot_date") \
 .agg(
     F.sum("q_members_t").alias("total_members"),
     F.count("*").alias("total_rows"),
     F.countDistinct("Branch").alias("distinct_branches"),
     F.countDistinct("Region").alias("distinct_regions")
 ).orderBy("CM_snapshot_date") \
 .show(truncate=False)

# Check MemSectorType null stability around April 2022
print("MemSectorType Null Stability: Q1-Q2 2022:")
print("-" * 70)
df.filter(
    F.col("CM_snapshot_date").isin(
        "2022-01-01", "2022-02-01", "2022-03-01",
        "2022-04-01", "2022-05-01", "2022-06-01"
    )
).groupBy("CM_snapshot_date") \
 .agg(
     F.sum(F.when(F.col("MemSectorType").isNull(), 1).otherwise(0))
      .alias("sector_nulls"),
     F.sum(F.when(F.col("YoJ").isNull(), 1).otherwise(0))
      .alias("yoj_nulls"),
     F.countDistinct("CatName").alias("distinct_catnames")
 ).orderBy("CM_snapshot_date") \
 .show(truncate=False)

# Check for any new cleaning flags appearing in April 2022
print("Cleaning Flag Distribution: Q1-Q2 2022:")
print("-" * 70)
df.filter(
    F.col("CM_snapshot_date").isin(
        "2022-01-01", "2022-02-01", "2022-03-01",
        "2022-04-01", "2022-05-01", "2022-06-01"
    )
).groupBy("CM_snapshot_date", "cleaning_flag") \
 .agg(F.count("*").alias("record_count")) \
 .orderBy("CM_snapshot_date", F.col("record_count").desc()) \
 .show(50, truncate=False)

Membership Count Stability: Q1-Q2 2022:
----------------------------------------------------------------------
+----------------+------------------+----------+-----------------+----------------+
|CM_snapshot_date|total_members     |total_rows|distinct_branches|distinct_regions|
+----------------+------------------+----------+-----------------+----------------+
|2022-01-01      |1480516.627084682 |290027    |100              |13              |
|2022-02-01      |1477220.9026191395|291396    |100              |13              |
|2022-03-01      |1477114.0142581237|292591    |100              |13              |
|2022-04-01      |1478274.9406240007|294946    |100              |13              |
|2022-05-01      |1477093.2304101936|294155    |100              |13              |
|2022-06-01      |1477116.9833793337|294584    |100              |13              |
+----------------+------------------+----------+-----------------+----------------+

MemSectorType Null Stability: Q1-Q2 2022:
------

**RESULT**

The broader anomaly check confirms that the April 2022 `YoB` null spike is isolated and does not reflect a wider system disruption. Membership counts remain stable across Q1-Q2 2022, ranging from 1.477M to 1.480M with no significant step change. Distinct branch and region counts hold constant at 100 and 13 respectively throughout the period. `MemSectorType` nulls, `YoJ` nulls, and cleaning flag distributions are all consistent across the six-month window with no new anomalies introduced in April 2022.

One minor observation is a step increase in distinct `CatName` values from 14 to 16 in June 2022, representing the introduction of two new membership subcategories: `Nurse - Career Break` and `Nursing Support Worker - Career Break`. These Career Break categories were introduced alongside the existing `Voluntary Break` variants, which were subsequently phased out by May 2023 as the terminology standardised around Career Break. A third new subcategory, `Nursing Support Worker - Student Nursing Associate`, was introduced in February 2025 through a gradual progressive rollout, representing a genuinely new membership pathway for nursing associates. This is noted for reference and will be examined further in the Exploratory Data Analysis notebooks.

The April 2022 anomaly is therefore confirmed as specific to birth year data capture only, with no evidence of a broader system migration or data quality failure across other dimensions in that period. Combined with the May 2024 `MemSectorType` label transition, the evidence points to a series of targeted, incremental system updates by the organisation over the observation period rather than a single wholesale migration event.

**Status:** ⚠️ Investigate

#### 2.3.3 April 2022 New vs Existing Member Check

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.3.3 April 2022 New vs Existing Member Check
# ═══════════════════════════════════════════════════════════════

# YoJ distribution for April 2022 null YoB records
print("YoJ Distribution for April 2022 Null YoB Records:")
print("-" * 70)
df.filter(
    (F.col("CM_snapshot_date") == "2022-04-01") &
    F.col("YoB").isNull()
).groupBy("YoJ") \
 .agg(F.count("*").alias("record_count"),
      F.sum("q_members_t").alias("total_members")) \
 .orderBy(F.col("record_count").desc()) \
 .show(50, truncate=False)

# Summary: new vs existing members
print("New vs Existing Member Summary (April 2022 Null YoB):")
print("-" * 70)
df.filter(
    (F.col("CM_snapshot_date") == "2022-04-01") &
    F.col("YoB").isNull()
).withColumn(
    "member_type",
    F.when(F.col("YoJ") == 2022, F.lit("New Member (joined 2022)"))
     .when(F.col("YoJ") >= 2020, F.lit("Recent Member (joined 2020-2021)"))
     .otherwise(F.lit("Existing Member (joined pre-2020)"))
).groupBy("member_type") \
 .agg(F.count("*").alias("record_count"),
      F.sum("q_members_t").alias("total_members")) \
 .orderBy(F.col("record_count").desc()) \
 .show(truncate=False)

YoJ Distribution for April 2022 Null YoB Records:
----------------------------------------------------------------------
+----+------------+------------------+
|YoJ |record_count|total_members     |
+----+------------+------------------+
|2021|556         |2966.1520190023552|
|2020|371         |1784.441805225659 |
|2019|311         |1193.586698337296 |
|1988|305         |1487.5296912114059|
|1987|292         |1454.8693586698375|
|1986|282         |1300.4750593824263|
|1985|279         |1386.579572446559 |
|1982|263         |1199.5249406175801|
|1984|254         |1220.3087885985776|
|2018|237         |878.8598574821873 |
|1983|232         |1012.4703087886011|
|1981|226         |929.3349168646101 |
|2001|188         |724.4655581947751 |
|2017|181         |635.3919239904997 |
|1980|175         |638.3610451306422 |
|1999|160         |691.8052256532071 |
|2000|158         |638.3610451306417 |
|2003|155         |564.1330166270789 |
|1995|153         |685.8669833729222 |
|1996|153         |65

**RESULT**

The `YoJ` distribution for April 2022 null `YoB` records confirms that 86.8% belong to existing members who joined before 2020, with join years spanning as far back as the 1970s. Only 1.3% are new members who joined in 2022, definitively ruling out new member intake as the primary driver of the null spike.

The concentration of nulls in long-standing members with `YoJ` values from the 1980s and 1990s is consistent with a cohort of members whose birth year was never captured at the point of original registration, a period when digital record keeping in membership organisations was less standardised. The elevated counts for `YoJ = 2021` and `YoJ = 2020` suggest that pandemic-period joiners also had birth year capture disrupted due to administrative pressures during that period.

Combined with the `YOB_DEFAULT_PLACEHOLDER` cleaning flag analysis from Section 2.3.1, the evidence points to a batch processing event in April 2022 that touched existing long-standing member records, surfacing a pre-existing data gap in birth year coverage rather than introducing new missingness. This further supports the MAR classification for `YoB` established in Section 2.2, as the missingness is systematically related to membership vintage rather than the birth year value itself.

**Status:** ⚠️ Investigate

### 2.4 Missingness Impact Assessment

**CONTEXT**

Having established the overall missingness profile, formal classifications, and temporal patterns, the final step in the missingness analysis is to quantify the analytical impact of missing values on the core metrics that will drive subsequent notebooks. Specifically, understanding what proportion of total membership units `SUM(q_members_t)` is affected by missingness in key analytical dimensions, such as `YoB` and `MemSectorType`, determines how confidently downstream segmentation and cohort analysis can be interpreted.

**PURPOSE**

To ensure:
1. The analytical impact of missingness on core membership metrics is quantified
2. Downstream notebooks are aware of the proportion of membership units excluded from each analytical dimension
3. Missingness limitations are documented transparently for the analysis methodology chapter
4. A missingness impact summary is available as a reference for all subsequent analytical decisions

**STEP**

For each column with non-zero missingness, calculate the proportion of total `SUM(q_members_t)` affected by null values. Compare the record-level null percentage against the membership-unit-level null percentage to determine whether missingness is concentrated in high or low membership segments. Produce a consolidated impact summary table.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.4 Missingness Impact Assessment
# ═══════════════════════════════════════════════════════════════

# Total membership units across entire dataset
total_members = df.agg(F.sum("q_members_t")).collect()[0][0]
total_rows = df.count()

print(f"Total membership units : {total_members:,.2f}")
print(f"Total rows             : {total_rows:,}")
print("-" * 70)

# Columns to assess (excluding q_leavers_t and flag columns)
impact_cols = ["YoB", "YoJ", "MemSectorType"]

impact_data = []
for col_name in impact_cols:
    null_rows = df.filter(F.col(col_name).isNull()).count()
    null_members = df.filter(F.col(col_name).isNull()) \
                     .agg(F.sum("q_members_t")).collect()[0][0]
    row_pct = round(null_rows / total_rows * 100, 4)
    member_pct = round(null_members / total_members * 100, 4)
    impact_data.append((col_name, null_rows, row_pct, 
                        round(null_members, 2), member_pct))

impact_df = spark.createDataFrame(
    impact_data,
    ["column", "null_rows", "row_null_pct", 
     "null_members", "member_null_pct"]
)

print("Missingness Impact Assessment:")
print("-" * 70)
impact_df.show(truncate=False)

print("Interpretation:")
print("-" * 70)
for row in impact_df.collect():
    diff = round(row["member_null_pct"] - row["row_null_pct"], 4)
    direction = "higher" if diff > 0 else "lower"
    print(f"{row['column']}: membership impact ({row['member_null_pct']}%) "
          f"is {abs(diff)}% {direction} than row impact ({row['row_null_pct']}%)")

Total membership units : 96,940,014.84
Total rows             : 18,461,480
----------------------------------------------------------------------
Missingness Impact Assessment:
----------------------------------------------------------------------
+-------------+---------+------------+------------+---------------+
|column       |null_rows|row_null_pct|null_members|member_null_pct|
+-------------+---------+------------+------------+---------------+
|YoB          |275407   |1.4918      |1157713.78  |1.1943         |
|YoJ          |39       |2.0E-4      |115.8       |1.0E-4         |
|MemSectorType|1436695  |7.7821      |4468711.4   |4.6098         |
+-------------+---------+------------+------------+---------------+

Interpretation:
----------------------------------------------------------------------
YoB: membership impact (1.1943%) is 0.2975% lower than row impact (1.4918%)
YoJ: membership impact (0.0001%) is 0.0001% lower than row impact (0.0002%)
MemSectorType: membership impact (4.

**RESULT**

The missingness impact assessment confirms that null values in key analytical columns have a proportionally lower impact on membership units than on row counts, indicating that missingness is concentrated in smaller cohort segments rather than high-membership segments.

`YoB` null rows represent 1.49% of records but only 1.19% of membership units, a difference of 0.30 percentage points, confirming that members with missing birth years tend to belong to smaller cohort segments. Age-based cohort analysis will therefore operate on 98.81% of total membership units, a robust analytical base.

`MemSectorType` shows the most significant divergence, with null rows at 7.78% of records but only 4.61% of membership units, a difference of 3.17 percentage points. This confirms that the non-active employment categories driving the majority of `MemSectorType` nulls, such as retired and career break members, are represented in smaller cohort segments. Sector-based segmentation analysis will operate on 95.39% of total membership units.

`YoJ` missingness is negligible at 0.0001% of membership units, carrying no meaningful analytical impact.

These findings confirm that while missingness exists across several dimensions, its impact on the core membership unit metric is consistently lower than the record-level null rates suggest, strengthening confidence in the analytical validity of all subsequent notebooks.

**Status:** ✓ Pass

## 3.0 Summary & Conclusions

### 3.1 Missingness Summary

**SUMMARY**

Notebook 03 conducted a systematic four-stage missingness analysis across the Silver table, establishing a comprehensive understanding of null patterns, their causes, and their analytical implications prior to exploratory analysis.

The overall missingness profile confirmed four columns with non-zero null rates across 12 analytical columns. `q_leavers_t` retains its 98.90% null rate by design, `MemSectorType` carries a 7.78% structural null rate, `YoB` carries a 1.49% null rate resulting from the Notebook 02 cleaning pipeline, and `YoJ` is negligible at 0.0002%.

Formal MCAR/MAR/MNAR classification confirmed that no columns carry MNAR missingness, indicating no evidence of self-selection bias in the dataset. `q_leavers_t` and `YoJ` are classified as MCAR, `YoB` and `MemSectorType` as MAR and Not Applicable respectively. These classifications will govern the analytical treatment of missing values across all subsequent notebooks.

Temporal analysis revealed a notable spike in `YoB` nulls in April 2022, increasing from approximately 4,900 to 7,812 records in a single month before returning to the expected range in May 2022. Deep dive investigation confirmed the spike is driven by existing long-standing members with join years spanning the 1970s to 1990s, consistent with a batch processing event surfacing pre-existing birth year data gaps rather than introducing new missingness. The broader anomaly check confirmed no other dimensions were affected, isolating the event to birth year data capture only.

The missingness impact assessment confirmed that null values have a proportionally lower impact on membership units than on row counts across all affected columns, with `MemSectorType` showing the largest divergence at 7.78% of rows versus 4.61% of membership units. Downstream analysis will operate on a minimum of 95.39% of total membership units across all analytical dimensions, confirming the dataset remains analytically robust despite identified missingness.

### 3.2 Next Steps

Notebook 04: Exploratory Data Analysis Part 1 will build directly on the validated, classified missingness findings established in this notebook. The following analytical workstreams are planned:

**Temporal Analysis**
Examine membership growth and churn trends across the 60-month observation period from January 2021 to December 2025. Identify seasonal patterns, year-over-year changes, and any structural breaks in the membership time series.

**Distributional Analysis**
Profile the distribution of key numerical variables including `YoB`, `YoJ`, `q_members_t`, and `q_leavers_t`. Identify skewness, kurtosis, and outlier patterns that may influence downstream statistical analysis.

**CatName Investigation**
Examine the two new `CatName` categories introduced in June 2022 identified during the temporal missingness analysis in Section 2.3.2, establishing their membership composition and relevance to churn behaviour.

**Null Exclusion Strategy**
Apply the MCAR/MAR/MNAR classifications from Section 2.2 consistently across all analytical dimensions, explicitly excluding null `YoB` rows from age-based analysis and retaining null `MemSectorType` rows with appropriate labelling in sector-based aggregations.

### Run Metadata

**Notebook:** 03 - Missingness & Data Quality Validation 
**Author:** D. 
**Executed:** {RUN_TS} 
**Dataset:** churn_t_db.csv (18,461,480 rows, 14 columns) 
**Source:** Silver table at /Volumes/workspace/rcn_churn/silver/churn_cleaned/ 
**Status:** ✓ Complete

In [0]:
# ═══════════════════════════════════════════════════════════════
# Notebook 03 Complete
# ═══════════════════════════════════════════════════════════════

from datetime import datetime
RUN_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("=" * 70)
print("Notebook 03: Missingness & Data Quality Validation")
print("=" * 70)
print(f"Status    : Complete")
print(f"Executed  : {RUN_TS}")
print("=" * 70)

Notebook 03: Missingness & Data Quality Validation
Status    : Complete
Executed  : 2026-03-12 16:11:12
